In [ ]:
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
from scipy.ndimage import gaussian_filter1d
import os
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error


In [ ]:
DATA_DIR = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == 'v8' else os.getcwd()
xlsx = pd.ExcelFile(os.path.join(DATA_DIR, 'Data for Datathon (Revised).xlsx'))
template = pd.read_csv(os.path.join(DATA_DIR, 'template_forecast_v00.csv'))

portfolios = ['A','B','C','D']

daily = {}
for p in portfolios:
    df = pd.read_excel(xlsx, f'{p} - Daily')
    df['Date'] = pd.to_datetime(df['Date'].str.strip().str.rsplit(' ', n=1).str[0], format='%m/%d/%y')
    df = df.sort_values('Date').reset_index(drop=True)
    df.columns = [c.strip() for c in df.columns]
    daily[p] = df
print({p: len(daily[p]) for p in portfolios})

mmap = {'January':1,'February':2,'March':3,'April':4,'May':5,'June':6,
        'July':7,'August':8,'September':9,'October':10,'November':11,'December':12}
intervals = {}
for p in portfolios:
    df = pd.read_excel(xlsx, f'{p} - Interval')
    df.columns = [c.strip() for c in df.columns]
    df = df.dropna(subset=['Interval']).copy()
    df['mnum'] = df['Month'].map(mmap)
    df['Day'] = df['Day'].astype(int)
    df['Date'] = pd.to_datetime(dict(year=2024, month=df['mnum'], day=df['Day']))
    df['slot'] = df['Interval'].apply(lambda t: t.hour*2 + t.minute//30)
    df = df.sort_values(['Date','slot']).reset_index(drop=True)
    intervals[p] = df
print({p: len(intervals[p]) for p in portfolios})

staff = pd.read_excel(xlsx, 'Daily Staffing')
staff.columns = ['Date'] + [f'Staff_{p}' for p in portfolios]
staff['Date'] = pd.to_datetime(staff['Date'])
staff = staff.sort_values('Date').reset_index(drop=True)
print(f'staffing: {len(staff)} days')

In [ ]:
# features
# v7: removed abandon rate lags and CCT cross-lags for CV model
# permutation importance showed they were hurting predictions

holidays = pd.to_datetime(['2024-01-01','2024-01-15','2024-02-19','2024-05-27','2024-06-19',
    '2024-07-04','2024-09-02','2024-10-14','2024-11-11','2024-11-28','2024-12-25',
    '2025-01-01','2025-01-20','2025-02-17','2025-05-26','2025-06-19',
    '2025-07-04','2025-09-01','2025-10-13','2025-11-11','2025-11-27','2025-12-25'])

def make_features(df, port):
    f = pd.DataFrame()
    f['Date'] = df['Date']
    f['dow'] = df['Date'].dt.dayofweek
    f['dom'] = df['Date'].dt.day
    f['month'] = df['Date'].dt.month
    f['woy'] = df['Date'].dt.isocalendar().week.astype(int)
    f['year'] = df['Date'].dt.year
    f['wknd'] = (f['dow'] >= 5).astype(int)
    f['mon'] = (f['dow'] == 0).astype(int)
    f['dow_s'] = np.sin(2*np.pi*f['dow']/7)
    f['dow_c'] = np.cos(2*np.pi*f['dow']/7)
    f['month_s'] = np.sin(2*np.pi*f['month']/12)
    f['month_c'] = np.cos(2*np.pi*f['month']/12)
    f['holiday'] = df['Date'].isin(holidays).astype(int)
    f['month_start'] = (f['dom'] <= 5).astype(int)
    f['month_end'] = (f['dom'] >= 26).astype(int)
    
    # only keep lags for the same metric type
    # abandon rate lags were hurting CV model, CCT lags were noise for CV
    if 'Call Volume' in df.columns:
        f['Call Volume_l7'] = df['Call Volume'].shift(7)
        f['Call Volume_l28'] = df['Call Volume'].shift(28)
        f['Call Volume_l365'] = df['Call Volume'].shift(365)
        f['Call Volume_r7'] = df['Call Volume'].rolling(7).mean()
        f['Call Volume_r30'] = df['Call Volume'].rolling(30).mean()
        f['Call Volume_ew'] = df['Call Volume'].ewm(span=7).mean()
    
    if 'CCT' in df.columns:
        f['CCT_l7'] = df['CCT'].shift(7)
        f['CCT_l14'] = df['CCT'].shift(14)
        f['CCT_l28'] = df['CCT'].shift(28)
        f['CCT_l365'] = df['CCT'].shift(365)
        f['CCT_ew'] = df['CCT'].ewm(span=7).mean()
    
    # staffing
    sc = f'Staff_{port}'
    f = f.merge(staff[['Date',sc]].rename(columns={sc:'agents'}), on='Date', how='left')
    
    for m in ['Call Volume','CCT','Abandon Rate']:
        if m in df.columns:
            f[f'tgt_{m}'] = df[m]
    return f

feats = {}
for p in portfolios:
    feats[p] = make_features(daily[p], p)
print({p: feats[p].shape for p in portfolios})

In [ ]:
# intraday profiles from the 3 month interval data
cv_prof, abd_prof, cct_prof = {}, {}, {}

for p in portfolios:
    df = intervals[p].copy()
    df['dow'] = df['Date'].dt.dayofweek
    dtot = df.groupby('Date')['Call Volume'].sum().reset_index()
    dtot.columns = ['Date','dtot']
    df = df.merge(dtot, on='Date')
    abt = df.groupby('Date')['Abandoned Calls'].transform('sum')
    df['cv_pct'] = df['Call Volume'] / df['dtot'].replace(0, np.nan)
    df['abd_pct'] = df['Abandoned Calls'] / abt.replace(0, np.nan)
    
    cv_prof[p], abd_prof[p], cct_prof[p] = {}, {}, {}
    for dow in range(7):
        sub = df[df['dow']==dow]
        for col, store in [('cv_pct',cv_prof), ('abd_pct',abd_prof)]:
            pr = sub.groupby('slot')[col].median()
            a = np.zeros(48)
            a[pr.index.astype(int)] = pr.values
            a = np.nan_to_num(a, 0)
            a = gaussian_filter1d(a, sigma=0.7)
            if a.sum() > 0: a /= a.sum()
            store[p][dow] = a
        
        pr = sub.groupby('slot')['CCT'].median()
        a = np.zeros(48)
        a[pr.index.astype(int)] = pr.values
        a = np.nan_to_num(a, 0)
        a = gaussian_filter1d(a, sigma=0.7)
        cct_prof[p][dow] = a

# avg CCT during profile period for scaling
prof_cct_avg = {}
for p in portfolios:
    msk = (daily[p]['Date']>='2024-04-01') & (daily[p]['Date']<='2024-06-30')
    prof_cct_avg[p] = daily[p].loc[msk, 'CCT'].mean()

print('done')


In [ ]:
# train + predict
# v8: quantile=0.50 (median), ml_weight=0.80, no bias
# tested on validation: this combo gives lowest volume MAE

def feat_cols(df):
    skip = ['Date','tgt_Call Volume','tgt_CCT','tgt_Abandon Rate']
    return [c for c in df.columns if c not in skip]

preds = {}
aug = pd.date_range('2025-08-01','2025-08-31')

for p in portfolios:
    print(f'\n--- {p} ---')
    ft = feats[p]
    cols = feat_cols(ft)
    ok = ft[cols].notna().all(axis=1) & ft['tgt_Call Volume'].notna() & ft['tgt_CCT'].notna()
    cl = ft[ok].copy()
    
    trn = cl['Date'] < '2025-07-01'
    val = (cl['Date']>='2025-07-01') & (cl['Date']<'2025-08-01')
    Xtr, Xv = cl.loc[trn, cols].values, cl.loc[val, cols].values
    
    summer = cl['Date'].dt.month.isin([5,6,7,8,9,10])
    trn_s = trn & summer
    print(f'  full: {trn.sum()}, summer: {trn_s.sum()}')
    
    preds[p] = {}
    d = daily[p]
    a24 = d[(d['Date'].dt.month==8)&(d['Date'].dt.year==2024)]
    
    for met in ['Call Volume','CCT']:
        ytr = cl.loc[trn, f'tgt_{met}'].values
        yv = cl.loc[val, f'tgt_{met}'].values
        ytr_s = cl.loc[trn_s, f'tgt_{met}'].values
        Xtr_s = cl.loc[trn_s, cols].values
        
        # v8: quantile=0.50 for volume (pure median, no upward push)
        # since we removed bias, dont need quantile to compensate
        q = 0.50 if met=='Call Volume' else 0.52
        
        gb = HistGradientBoostingRegressor(loss='quantile', quantile=q,
            max_iter=500, max_depth=6, learning_rate=0.05,
            min_samples_leaf=10, random_state=42)
        gb.fit(Xtr, ytr)
        
        has_summer = len(ytr_s) > 20
        if has_summer:
            gb_s = HistGradientBoostingRegressor(loss='quantile', quantile=q,
                max_iter=400, max_depth=5, learning_rate=0.05,
                min_samples_leaf=8, random_state=42)
            gb_s.fit(Xtr_s, ytr_s)
        
        rd = Ridge(alpha=1.0)
        rd.fit(np.nan_to_num(Xtr,0), ytr)
        
        vp_full = 0.6*gb.predict(Xv) + 0.4*rd.predict(np.nan_to_num(Xv,0))
        mae_full = mean_absolute_error(yv, vp_full)
        
        best_w, best_mae = 0, mae_full
        if has_summer:
            vp_s = gb_s.predict(Xv)
            for w in [0, 0.15, 0.25, 0.35, 0.5]:
                m = mean_absolute_error(yv, (1-w)*vp_full + w*vp_s)
                if m < best_mae:
                    best_w, best_mae = w, m
        print(f'  {met} — full:{mae_full:.2f} best_w:{best_w} MAE:{best_mae:.2f}')
        
        amsk = (ft['Date']>='2025-08-01') & (ft['Date']<='2025-08-31')
        Xa = ft.loc[amsk, cols].fillna(method='ffill').fillna(method='bfill').fillna(0)
        ml_full = 0.6*gb.predict(Xa.values) + 0.4*rd.predict(np.nan_to_num(Xa.values,0))
        if has_summer and best_w > 0:
            ml = (1-best_w)*ml_full + best_w*gb_s.predict(Xa.values)
        else:
            ml = ml_full
        
        # v8: 80% ML, 20% baseline (was 70/30)
        g24 = d[(d['Date'].dt.year==2024)&(d['Date'].dt.month<=7)][met].mean()
        g25 = d[(d['Date'].dt.year==2025)&(d['Date'].dt.month<=7)][met].mean()
        gr = g25/g24 if g24>0 else 1.0
        bl = np.zeros(31)
        for i,dt in enumerate(aug):
            m = a24[a24['Date'].dt.dayofweek==dt.dayofweek][met].values
            bl[i] = m.mean()*gr if len(m)>0 else a24[met].mean()*gr
        
        final = 0.8*ml[:31] + 0.2*bl
        preds[p][met] = final
        print(f'  aug avg: {final.mean():.1f}')
    
    recent = d[(d['Date']>='2025-06-01')&(d['Date']<'2025-08-01')]
    abd = np.zeros(31)
    for i,dt in enumerate(aug):
        dw = dt.dayofweek
        r = recent[recent['Date'].dt.dayofweek==dw]['Abandon Rate']
        a = a24[a24['Date'].dt.dayofweek==dw]['Abandon Rate']
        if len(r)>0 and len(a)>0:
            abd[i] = 0.6*r.mean() + 0.4*a.mean()
        elif len(r)>0:
            abd[i] = r.mean()
        else:
            abd[i] = d['Abandon Rate'].tail(60).mean()
    abd *= 1.1
    abd = np.clip(abd, 0.002, 0.25)
    preds[p]['Abandon Rate'] = abd
    abd_avg = a24['Abandon Rate'].mean()
    print(f'  ABD: {abd.mean():.4f} (aug24 was {abd_avg:.4f})')

print('\ndone')

In [ ]:
# split daily preds into 30min intervals
aug_dates = pd.date_range('2025-08-01','2025-08-31')
res = {p: {'cv':[],'abd':[],'ar':[],'cct':[]} for p in portfolios}

for p in portfolios:
    dcv = preds[p]['Call Volume']
    dcct = preds[p]['CCT']
    dar = preds[p]['Abandon Rate']
    dabd = dcv * dar
    
    for i,dt in enumerate(aug_dates):
        dw = dt.dayofweek
        cv = dcv[i] * cv_prof[p][dw]
        ab = dabd[i] * abd_prof[p][dw]
        sc = dcct[i] / prof_cct_avg[p] if prof_cct_avg[p]>0 else 1.0
        cc = cct_prof[p][dw] * sc
        ar = np.where(cv>0, ab/cv, 0.0)
        res[p]['cv'].append(cv)
        res[p]['abd'].append(ab)
        res[p]['ar'].append(ar)
        res[p]['cct'].append(cc)

for p in portfolios:
    for k in res[p]:
        res[p][k] = np.array(res[p][k])


In [ ]:
# v8: no bias - volume accuracy matters more than workload penalty
# tested this on validation, bias=1.0 gives lowest MAE

for p in portfolios:
    res[p]['cv'] = np.clip(res[p]['cv'], 0, None)
    res[p]['abd'] = np.clip(res[p]['abd'], 0, None)
    res[p]['cct'] = np.clip(res[p]['cct'], 0, None)
    bad = res[p]['abd'] > res[p]['cv']
    res[p]['abd'][bad] = res[p]['cv'][bad]
    cv, ab = res[p]['cv'], res[p]['abd']
    res[p]['ar'] = np.clip(np.where(cv>0, ab/cv, 0.0), 0, 1)
    res[p]['cv'] = np.round(cv).astype(int)
    res[p]['abd'] = np.round(res[p]['abd']).astype(int)

In [ ]:
# save
rows = []
for day in range(31):
    for slot in range(48):
        h, m = slot//2, (slot%2)*30
        row = {'Month':'August', 'Day':str(day+1), 'Interval':f'{h}:{m:02d}'}
        for p in portfolios:
            row[f'Calls_Offered_{p}'] = int(res[p]['cv'][day,slot])
            row[f'Abandoned_Calls_{p}'] = int(res[p]['abd'][day,slot])
            row[f'Abandoned_Rate_{p}'] = round(float(res[p]['ar'][day,slot]), 6)
            row[f'CCT_{p}'] = round(float(res[p]['cct'][day,slot]), 2)
        rows.append(row)

sub = pd.DataFrame(rows)[template.columns.tolist()]
out = os.path.join(os.getcwd(), 'forecast_v8.csv')
sub.to_csv(out, index=False)

assert sub.shape == (1488, 19)
assert not sub.isnull().any().any()
for p in portfolios:
    assert (sub[f'Calls_Offered_{p}']>=0).all()
    assert (sub[f'Abandoned_Calls_{p}']<=sub[f'Calls_Offered_{p}']).all()
print('all good')

for p in portfolios:
    a24 = daily[p][(daily[p]['Date'].dt.month==8)&(daily[p]['Date'].dt.year==2024)]
    fc = sub[f'Calls_Offered_{p}'].sum()
    ac = a24['Call Volume'].sum()
    print(f'{p}: {fc:,} vs {ac:,.0f} ({fc/ac:.3f})')